### Objetivo da Atividade

- Colocar em prática os conceitos de Ciência de Dados, Aprendizagem de
Máquina, Classificação, Redes Neurais, Função ReLU e Overfitting vistos em
aula.
- Você deverá utilizar um dataset real do Brasil, realizar o pré-processamento
dos dados, treinar uma Rede Neural Artificial (Deep Learning) e interpretar
os resultados obtidos.

### Etapa 1: Importação e Exploração Inicial

- Tarefa:
    - Observe as colunas relacionadas a datas de compra e entrega
    - Identifique quais informações podem indicar atraso na entrega
- Exemplo de colunas importantes:
    - order_purchase_timestamp
    - order_estimated_delivery_date
    - order_delivered_customer_date

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv("olist_orders_dataset.csv")
display(df)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [3]:
colunas_data = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

for col in colunas_data:
    df[col] = pd.to_datetime(df[col])

### Etapa 2: Criação da Variável Alvo (Classificação)

- Crie uma nova coluna chamada atrasou, onde:
    - 0 → Pedido entregue dentro do prazo
    - 1 → Pedido entregue com atraso
    - A coluna atrasou será a classe que o modelo irá prever.

In [4]:
df['atraso'] = np.where(df['order_delivered_customer_date'] > df['order_estimated_delivery_date'], 1, 0)

df['dias_estimados'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.days
df['dias_aprovacao'] = (df['order_approved_at'] - df['order_purchase_timestamp']).dt.total_seconds() / 86400
df['dias_para_transportadora'] = (df['order_delivered_carrier_date'] - df['order_approved_at']).dt.total_seconds() / 86400

df_clean = df.dropna(subset=['dias_estimados', 'dias_aprovacao', 'dias_para_transportadora', 'order_delivered_customer_date']).copy()

X = df_clean[['dias_estimados', 'dias_aprovacao', 'dias_para_transportadora']]
y = df_clean['atraso']

### Etapa 3: Pré-processamento e Transformação

- Converta as colunas de data para o tipo datetime
    - Crie pelo menos 1 variável explicativa, como:
    - Tempo estimado de entrega (em dias)
    - Trate valores ausentes (caso existam)
- Dica: comece com poucas variáveis para facilitar a interpretação.

### Etapa 4: Definição de X e y

- X → Variáveis de entrada (features)
- y → Variável alvo (atrasou)
- Divida os dados em:
    - Treino (80%)
    - Teste (20%)

### Etapa 6: Treinamento do Modelo

- Utilize a função de perda binary_crossentropy
- Utilize o otimizador Adam
- Treine o modelo por várias épocas (ex: 30 ou 40)
- Durante o treinamento, observe:
    - Loss de treino
    - Loss de validação


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rede_neural = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', max_iter=500, random_state=42)
rede_neural.fit(X_train_scaled, y_train)

previsoes_treino = rede_neural.predict(X_train_scaled)
previsoes_teste = rede_neural.predict(X_test_scaled)

### Etapa 7: Visualização do Aprendizado (Overfitting)

- Gere um gráfico contendo:
    - Curva do erro de treino
    - Curva do erro de validação
-  Analise se ocorre overfitting (quando a rede aprende bem o treino, mas piora na validação).

In [6]:
rede_iterativa = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', random_state=42, max_iter=1, warm_start=True)

erros_treino = []
erros_validacao = []
classes = np.unique(y_train)

for epoca in range(100):
    rede_iterativa.partial_fit(X_train_scaled, y_train, classes=classes)
    prob_treino = rede_iterativa.predict_proba(X_train_scaled)
    prob_validacao = rede_iterativa.predict_proba(X_test_scaled)
    erros_treino.append(log_loss(y_train, prob_treino))
    erros_validacao.append(log_loss(y_test, prob_validacao))

fig = go.Figure()

fig.add_trace(go.Scatter(
    y=erros_treino,
    mode ='lines',
    name='Erro no Treino',
    line=dict(color='#1f77b4', width=3)
))

fig.add_trace(go.Scatter(
    y=erros_validacao,
    mode='lines',
    name='Erro na Validação',
    line=dict(color='#ff7f0e', width=3)
))

fig.update_layout(
    title='Curva de Aprendizado: Avaliando Overfitting',
    xaxis_title='Épocas (Iterações)',
    yaxis_title='Erro (Log Loss)',
    template='plotly_white',
    legend=dict(yanchor="top",  y=0.99, xanchor="right", x=0.99),
    hovermode='x unified'
)

fig.show()

### Etapa 8: Avaliação do Modelo

- Calcule a acurácia no conjunto de teste
-  Interprete o resultado

In [7]:
print(f"Acurácia no Treino: {accuracy_score(y_train, previsoes_treino):.4f}")
print(f"Acurácia no Teste: {accuracy_score(y_test, previsoes_teste):.4f}")
print("\nRelatório de Classificação (Teste):")
print(classification_report(y_test, previsoes_teste))

Acurácia no Treino: 0.9265
Acurácia no Teste: 0.9279

Relatório de Classificação (Teste):
              precision    recall  f1-score   support

           0       0.93      1.00      0.96     17746
           1       0.84      0.13      0.22      1547

    accuracy                           0.93     19293
   macro avg       0.88      0.56      0.59     19293
weighted avg       0.92      0.93      0.90     19293



### Interpretação e Análise Crítica

Responda às questões abaixo:
1. O modelo apresentou overfitting? Como você identificou isso?
    - Resposta: Não, pois o a acurácia de treino e de testes ficaram testes ficaram praticamente iguais.
2. A Função ReLU contribui para o aprendizado da rede? Por quê?
    - Resposta: Sim, pois zera os cálculos de ativação negativos e lineariza os positivos. Sendo assim, isso acaba acelerando o processamento matemático.
3. Acurácia alta significa que o modelo é confiável neste contexto?
    - Resposta: Não, observando os dados, a maioria está desbalanceada, pois a grande parte chega no prazo.
4. Esse problema poderia ser resolvido com um modelo mais simples? Explique.
    - Resposta: Sim, usando Árvore de Decisão. Para dados tabulares lógicos estritos, árvores e redes neurais perfomam de forma muito similar, tendo a árvore a grande vamtagem de alta explicabilidade algoritmica.

Respostas dissertativas:
1. Qual variável foi escolhida como alvo e por quê?
    - Resposta: A variável alvo foi a classificação de atraso, sendo criada nos cálculos de (order_delivered_customer_date) e (order_estimated_delivery_date). Escolhi as duas pois são as duas que indicam o cumprimento de prazos.
2. Quais variáveis foram usadas como entrada na rede?
    - Resposta: timestamps que corresponde ao total de dias estimados para a entrega, o total de dias para aprovação do pagamento e o tempo de respasse logístico.
3. O modelo apresentou overfitting?
    - Respostas: Não
4. O que poderia ser feito para melhorar o desempenho?
    - Resposta: Poderia haver um equilíbrio a quantidade de atrasos com entregas no prazo.